# Pipeline step 1: stratified culture-vs-language diagnosis

Goal: measure the **culture** effect on jailbreak success (`us` / `japan` vs `neutral`)
**controlling for language**, instead of pooling `en`/`ja`/`mix` together. Language
becomes the stratifying ("control") variable; culture is the exposure of interest.

**Method: Cochran-Mantel-Haenszel (CMH).** For each language stratum, build a 2x2 table
(culture exposed vs not) x (unsafe vs not), then pool across strata into one combined
odds ratio. This is the standard epidemiology tool for exactly this situation — "does X
affect Y, after accounting for a third variable Z that might confound it" — pulled in
here instead of the plain pooled logistic regression from `stat_analysis.ipynb`, which
mixed language into the same model as culture rather than using it as a control.

If the per-language (`en`, `ja`) odds ratios roughly agree *and* the pooled MH estimate
is significant, that's good evidence of a real, language-independent cultural effect. If
they disagree a lot, the "culture" signal is partly a language artifact.

No GPU needed — this is pandas/scipy on the already-loaded `observations.csv`.


> **Migrated from the `ez-penta` research repo (experiment `post_train_2`).** The executed original with its outputs from the 2026-11-12 run is [`executed/pipeline1_stratified_diagnosis.ipynb`](executed/pipeline1_stratified_diagnosis.ipynb). This copy only changes where files are read and written (see `configs/experiment.json`).

In [ ]:
# Project setup (Colab or local): fetch this repository's code and small data files, then load
# the shared config. Large files come from the project's Hugging Face datasets on first use.
import subprocess
import sys
from pathlib import Path

if Path("/content").exists():                     # Google Colab
    REPO = Path("/content/tiny-aya-safety")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/ishane0620/tiny-aya-safety", str(REPO)], check=True)
else:                                             # local checkout: this notebook is in <repo>/notebooks
    REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from tiny_aya_safety.config import CONFIG, adapter_source, artifact, data_path, output_dir

OUT = output_dir()   # everything this notebook writes goes here (override with TAS_OUTPUT_DIR)
print("repo:", REPO, "| outputs:", OUT)

## 0. Load data (same Drive-detection as the other notebooks)

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# Dataset2 baseline evaluation: every prompt with Tiny Aya Water's original answer and the
# AuditKIT/Qwen3Guard verdict (published as the project's Dataset2 Hugging Face dataset).
observations = pd.read_parquet(data_path("observations.parquet"))
usable = observations[
    (observations["analysis_eligible"] == True)
    & (observations["generation_status"] == "completed")
    & (observations["scoring_status"] == "completed")
    & (observations["split"] != "heldout")
].copy()
harmful = usable[usable["kind"] == "harmful"].copy()
harmful["unsafe"] = harmful["safety"].eq("Unsafe").astype(int)
print(f"{len(harmful)} usable harmful-seed rows")

## 1. Mantel-Haenszel pooling function

Standard formulas (Mantel & Haenszel 1959; variance per Robins, Breslow & Greenland
1986): pooled odds ratio, MH chi-square test (with continuity correction), and a 95%
CI on the log odds ratio.

In [ ]:

def two_by_two(df, exposure_mask, outcome_col):
    a = int((exposure_mask & (df[outcome_col] == 1)).sum())
    b = int((exposure_mask & (df[outcome_col] == 0)).sum())
    c = int((~exposure_mask & (df[outcome_col] == 1)).sum())
    d = int((~exposure_mask & (df[outcome_col] == 0)).sum())
    return a, b, c, d


def mantel_haenszel(df, exposure_col, exposure_value, baseline_value, outcome_col, stratum_col):
    # Pool the (exposure_value vs baseline_value) odds ratio for `outcome_col` across
    # strata of `stratum_col`. Returns a dict with per-stratum and pooled results.
    strata = sorted(df[stratum_col].dropna().unique())
    per_stratum = []
    num_or, den_or = 0.0, 0.0  # for OR_MH
    sum_a, sum_Ea, sum_Var = 0.0, 0.0, 0.0  # for the MH chi-square
    sum_R, sum_S, sum_P_R, sum_P_S, sum_Q_R, sum_Q_S = (0.0,) * 6  # for RBG variance

    for stratum in strata:
        sub = df[(df[stratum_col] == stratum) & (df[exposure_col].isin([exposure_value, baseline_value]))]
        exposure_mask = sub[exposure_col] == exposure_value
        a, b, c, d = two_by_two(sub, exposure_mask, outcome_col)
        n = a + b + c + d
        if n == 0 or (a + c) == 0 or (b + d) == 0:
            continue
        or_stratum = (a * d) / (b * c) if b * c > 0 else np.inf
        p_stratum = stats.fisher_exact([[a, b], [c, d]])[1]
        per_stratum.append({"stratum": stratum, "a_exposed_unsafe": a, "b_exposed_safe": b,
                              "c_unexposed_unsafe": c, "d_unexposed_safe": d, "n": n,
                              "odds_ratio": or_stratum, "fisher_p": p_stratum})

        num_or += (a * d) / n
        den_or += (b * c) / n
        Ea = (a + b) * (a + c) / n
        Va = (a + b) * (c + d) * (a + c) * (b + d) / (n**2 * (n - 1)) if n > 1 else 0
        sum_a += a
        sum_Ea += Ea
        sum_Var += Va

        R, S = (a * d) / n, (b * c) / n
        P, Q = (a + d) / n, (b + c) / n
        sum_R += R
        sum_S += S
        sum_P_R += P * R
        sum_P_S += P * S
        sum_Q_R += Q * R
        sum_Q_S += Q * S

    or_mh = num_or / den_or if den_or > 0 else np.nan
    chi2_mh = (abs(sum_a - sum_Ea) - 0.5) ** 2 / sum_Var if sum_Var > 0 else np.nan
    p_mh = stats.chi2.sf(chi2_mh, df=1) if np.isfinite(chi2_mh) else np.nan

    # Robins-Breslow-Greenland variance of ln(OR_MH)
    if sum_R > 0 and sum_S > 0:
        var_ln_or = (sum_P_R / (2 * sum_R**2)) + ((sum_P_S + sum_Q_R) / (2 * sum_R * sum_S)) + (sum_Q_S / (2 * sum_S**2))
        se_ln_or = np.sqrt(var_ln_or)
        ci_lo = np.exp(np.log(or_mh) - 1.96 * se_ln_or)
        ci_hi = np.exp(np.log(or_mh) + 1.96 * se_ln_or)
    else:
        ci_lo = ci_hi = np.nan

    return {
        "exposure": f"{exposure_col}={exposure_value} vs {baseline_value}",
        "or_mh": or_mh, "ci_lo": ci_lo, "ci_hi": ci_hi,
        "chi2_mh": chi2_mh, "p_mh": p_mh,
        "per_stratum": pd.DataFrame(per_stratum),
    }


## 2. Culture effect, pooled across `en`/`ja`/`mix` (language as the control variable)

Two exposures tested: `culture=us` vs `neutral`, and `culture=japan` vs `neutral`.

In [ ]:

results = {}
for culture_value in ["us", "japan"]:
    result = mantel_haenszel(harmful, "culture", culture_value, "neutral", "unsafe", "prompt_language")
    results[culture_value] = result
    print(f"=== {result['exposure']}, pooled across prompt_language strata ===")
    print(f"OR_MH = {result['or_mh']:.3f}  (95% CI {result['ci_lo']:.3f}-{result['ci_hi']:.3f})"
          f"  chi2_MH={result['chi2_mh']:.2f}  p={result['p_mh']:.6f}")
    display(result["per_stratum"].round(4))
    print()


## 3. Does the culture effect replicate in `en` AND `ja` separately?

This is the actual "control for language" check: compare the per-stratum odds ratios
(from the table above) between `en` and `ja`. Consistent sign and similar magnitude in
both = a real cultural effect. The `mix` stratum is shown too, as a third (weaker,
code-switched) data point, but `en`/`ja` are the two that matter for this check.

In [ ]:

for culture_value, result in results.items():
    per_stratum = result["per_stratum"].set_index("stratum")
    print(f"--- culture={culture_value} vs neutral: per-language odds ratio and Fisher p ---")
    for lang in ["en", "ja", "mix"]:
        if lang in per_stratum.index:
            row = per_stratum.loc[lang]
            print(f"  {lang}: OR={row['odds_ratio']:.3f}  p={row['fisher_p']:.5f}  n={int(row['n'])}")
    en_or = per_stratum.loc["en", "odds_ratio"] if "en" in per_stratum.index else np.nan
    ja_or = per_stratum.loc["ja", "odds_ratio"] if "ja" in per_stratum.index else np.nan
    same_direction = (en_or > 1) == (ja_or > 1)
    print(f"  en and ja agree on direction (both > 1 or both < 1): {same_direction}")
    print()


## 4. Same check, restricted to `framing=rp` (the known worst framing)

The earlier interaction test found `culture x framing` significant overall. This
repeats the stratified-by-language check specifically within the `rp` (roleplay)
framing, to see whether the `us+rp` combination is consistent across languages too.

In [ ]:

rp_only = harmful[harmful["framing"] == "rp"]
print(f"{len(rp_only)} rows with framing=rp")

rp_results = {}
for culture_value in ["us", "japan"]:
    result = mantel_haenszel(rp_only, "culture", culture_value, "neutral", "unsafe", "prompt_language")
    rp_results[culture_value] = result
    print(f"\n=== framing=rp only: {result['exposure']}, pooled across language ===")
    print(f"OR_MH = {result['or_mh']:.3f}  (95% CI {result['ci_lo']:.3f}-{result['ci_hi']:.3f})"
          f"  chi2_MH={result['chi2_mh']:.2f}  p={result['p_mh']:.6f}")
    display(result["per_stratum"].round(4))


## 5. Save the pooled estimates for Notebook 5 (before/after validation)

Notebook 5 will recompute this exact statistic on the post-fix model's outputs and
compare against these baseline numbers.

In [ ]:

import json

baseline_summary = {
    culture_value: {"or_mh": result["or_mh"], "ci_lo": result["ci_lo"], "ci_hi": result["ci_hi"],
                     "p_mh": result["p_mh"]}
    for culture_value, result in results.items()
}
baseline_summary_rp = {
    culture_value: {"or_mh": result["or_mh"], "ci_lo": result["ci_lo"], "ci_hi": result["ci_hi"],
                     "p_mh": result["p_mh"]}
    for culture_value, result in rp_results.items()
}
out = {"overall": baseline_summary, "framing_rp_only": baseline_summary_rp}
out_path = OUT / "pipeline1_baseline_mh_stats.json"
out_path.write_text(json.dumps(out, indent=2, default=float))
print(f"Saved baseline MH statistics to {out_path}")
print(json.dumps(out, indent=2, default=float))
